In [ ]:
import marimo as mo
import os
import numpy as np
import pandas as pd
import scampi
from scampi.scampi import SCAMPI
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import seaborn as sns
from matplotlib.gridspec import GridSpec

import warnings
warnings.simplefilter("ignore")

colors = sns.color_palette("tab20")

In [ ]:
annotations = [
    "Walk very\nslow", 
    "Normal\nWalk", 
    "Nordic Walk", 
    "Run", 
    "Cycle", 
    "Run", 
    "Normal\nWalk", 
    "Soccer", 
    "Rope\nJump"
]

k_max = 100
window_size = 160

The 7-th motiflet appears between activities

<hr />

In [ ]:
def find_dominant_window_sizes(X, offset=0.05):
    """Determine the Window-Size using dominant FFT-frequencies."""
    fourier = np.absolute(np.fft.fft(X))
    freqs = np.fft.fftfreq(X.shape[0], 1)
    plt.plot(fourier)
    plt.show()

    coefs = []
    window_sizes = []

    for coef, freq in zip(fourier, freqs):
        if coef and freq > 0:
            coefs.append(coef)
            window_sizes.append(1 / freq)

    coefs = np.array(coefs)
    window_sizes = np.asarray(window_sizes, dtype=np.int64)

    idx = np.argsort(coefs)[::-1]
    return next(
        (
            int(window_size / 2)
            for window_size in window_sizes[idx]
            if window_size in range(20, int(X.shape[0] * offset))
        ),
        window_sizes[idx[0]],
    )

def find_dominant_window_sizes2(X, offset=0.05, fs=100, f_min=1e-2):
    """Determine the Window-Size using dominant FFT-frequencies."""
    X = X - X.mean()
    fourier = np.fft.rfft(X)
    freqs = np.fft.rfftfreq(X.shape[0], d=1/fs)

    fourier = np.abs(fourier) / X.shape[0]             # Normalize amplitude
    fourier[1:-1] *= 2                          # Double non-DC, non-Nyquist bins

    plt.plot(freqs,fourier)
    # plt.xlim(0, fs / 2)
    plt.semilogx()

    mask = freqs >= f_min
    dominant_freq = freqs[mask][np.argmax(fourier[mask])]
    print("dominant frequency", dominant_freq, "corresponding window", fs/dominant_freq)
    plt.axvline(dominant_freq, color="red")
    ws = int(round(fs / dominant_freq))
    return ws


def load_dataset(dataset, selection=None):
    desc_filename = f"../datasets/{dataset}/desc.txt"
    desc_file = []

    with open(desc_filename, 'r') as file:
        for line in file.readlines(): desc_file.append(line.split(","))

    df = []

    for idx, row in enumerate(desc_file):
        if selection is not None and idx not in selection: continue
        (ts_name, window_size), change_points = row[:2], row[2:]
        if len(change_points) == 1 and change_points[0] == "\n": change_points = list()
        path = f'../datasets/{dataset}/'

        if os.path.exists(path + ts_name + ".txt"):
            ts = np.loadtxt(fname=path + ts_name + ".txt", dtype=np.float64)
        else:
            ts = np.load(file=path + "data.npz")[ts_name]

        df.append((ts_name, int(window_size), np.array([int(_) for _ in change_points]), ts))

    return pd.DataFrame.from_records(df, columns=["name", "window_size", "change_points", "time_series"])


def load_pamap():
    dataset="PAMAP"
    selection = [126, 127, 128] # Outdoor
    df_data = load_dataset(dataset)

    ts_name = df_data["name"].iloc[selection]
    ts = df_data.time_series.iloc[selection]
    cps = df_data.change_points.iloc[selection[0]]

    X = np.zeros((len(ts.values), len(ts.values[0])))
    for i, data in enumerate(ts.values):
        X[i] = data

    cps = np.concatenate([[0], cps, [X.shape[1]]])

    series = pd.DataFrame(data=X, index=ts_name)
    series.rename(index={'PAMAP_Outdoor_Subject8_IMU_Shoe_X-Acc': 'Shoe X-Acc', 
                         'PAMAP_Outdoor_Subject8_IMU_Shoe_Y-Acc': 'Shoe Z-Acc', 
                         'PAMAP_Outdoor_Subject8_IMU_Shoe_Z-Acc': 'Shoe Y-Acc'}, inplace=True)

    ## series = series.iloc[0, :]
    series = series.loc[["Shoe X-Acc"]]
    return series, cps

def zeucl(x, y):
    return np.linalg.norm(znorm(x) - znorm(y))

def extent(ts, w, motiflet):
    e = 0
    for i in motiflet:
      for j in motiflet:
          if j > i:
              e = max(e, zeucl(ts[i:i+w], ts[j:j+w]))
    return e



def znorm(x):
    return (x - x.mean()) / x.std()


def plot_topn(topn: SCAMPI, changepoints: np.array, annotations, fname=None):
    import matplotlib.gridspec as gridspec

    if topn.top_N <= 10:
        colors = sns.color_palette("tab10")
    else:
        colors = sns.color_palette("tab20")

    nrows = 1 + int(np.ceil(topn.top_N / 3))
    fig = plt.figure(figsize=(12, 2*nrows))

    # Create a GridSpec with 4 rows and 3 columns
    gs = gridspec.GridSpec(nrows, 3, figure=fig, hspace=0.4, wspace=0.3)

    # Top row: single plot spanning all 3 columns
    ax_top = fig.add_subplot(gs[0, :])
    # ax_top.set_title("Top Plot")
    ax_top.plot(topn.series, c="lightgray")
    for t in changepoints:
        ax_top.axvline(t, linestyle="dotted", c="lightgray")
    ax_top.set_xticks([])
    ax_top.set_yticks([])
    ax_top.set_ylim(-180, 200)
    for annotation, (b, e) in zip(annotations, 
                                  zip(changepoints[:-1], changepoints[1:])):
        x = (b + e) / 2
        ax_top.annotate(annotation, xy=(x, 100), fontsize=10, ha="center")

    # Bottom 3x3 grid occupying rows 1-3

    toplot = []
    for rank, elbows in enumerate(topn.elbow_points):
        if len(elbows) == 0:
            continue

        k = elbows[0]
        hsm = topn.motiflets[k][rank]
        d = extent(topn.series, topn.motif_length, hsm)
        # d = topn.dists[k, rank]

        toplot.append((d, hsm))
    toplot = sorted(toplot, key=lambda x: x[0])


    axes = []
    # for i, res in enumerate(topn.found_motiflets):
    for i, (d, hsm) in enumerate(toplot):
        if i >= len(colors):
            break
        row = i // 3 + 1
        col = i % 3
        # dists, motif_sets, elbow_points = res
        ax = fig.add_subplot(gs[row, col])
        ax.axis("off")
        mlabels = dict()
        for j, t in enumerate(hsm):
            idx = np.arange(t, min(t + topn.motif_length, len(topn.series)))
            subseq = topn.series[idx]
            ax.plot(znorm(subseq), c=colors[i], alpha=0.3)
            ax_top.plot(idx, subseq, c=colors[i])
            segment = np.searchsorted(changepoints, t, side="right") - 1
            segment = np.clip(segment, 0, len(annotations) - 1)
            l = annotations[segment]

            mlabels[l] = mlabels.get(l, 0) + 1
            # plt.plot(idx, subseq, c=colors[iteration])
        activity_label = "\n".join([
            "{} (x{})".format(k.replace("\n", " "), v)
            for k, v in mlabels.items()
        ])
        ax.set_title(f"{i+1:02d}. {activity_label} (e={d:.2f})",
                     fontsize=10, ha="left", loc="left", va="top")
        axes.append(ax)

    plt.tight_layout(pad=0)

    if fname is not None:
        plt.savefig(fname, dpi=300, bbox_inches="tight")

    plt.show()


In [ ]:
series_df, changepoints = load_pamap()
series = series_df.values[0,:]
n = series.shape[0]

In [ ]:
k_max = 100
motif_length = 200
top_N = 12

scampi_topn = SCAMPI(
    "Human Activity",
    series,
    n_jobs=-1,
    backend="scampi",
    verbose=False,
    scampi_delta=0.1
)
fig, ax = scampi_topn.plot_dataset()

dists, candidates, elbow_points = scampi_topn.fit_k_elbow(
    k_max=k_max,
    motif_length=motif_length,
    top_N=top_N,
    scampi_top_n_strategy="mask",
    filter=True,
    plot_elbows=False,
    plot_motifs_as_grid=False,
)

In [ ]:
plot_topn(
    scampi_topn, changepoints, annotations
)

In [ ]:
scampi_topn.elbow_points

In [ ]:
k_max, motif_length, top_N = 100, 400, 12

scampi_long_topn = SCAMPI(
    "Human Activity",
    series,
    n_jobs=-1,
    backend="scampi",
    verbose=False,
    scampi_delta=0.1
)

dists, candidates, elbow_points = scampi_long_topn.fit_k_elbow(
    k_max=k_max,
    motif_length=motif_length,
    top_N=top_N,
    scampi_top_n_strategy="mask",
    plot_elbows=False,
    plot_motifs_as_grid=False,
)

In [ ]:
scampi_long_topn.elbow_points

In [ ]:
  for rank, elbows in enumerate(scampi_long_topn.elbow_points):
      k = elbows[0]
      hsm = scampi_long_topn.motiflets[k][rank]
      print(f"top-{rank + 1}: elbows={elbows}, masked_k={k}, motiflet[:5]={hsm[:5]}, size={len(hsm)}")



In [ ]:
plot_topn(
    scampi_long_topn,
    changepoints,
    annotations,
    fname=f"human-activity-w{scampi_long_topn.motif_length}.pdf"
)

In [ ]:
last_two = series[changepoints[-3]:]
k_max, motif_length, top_N = 100, 110, 3

scampi_last_two = SCAMPI(
    "Human Activity",
    last_two,
    n_jobs=-1,
    backend="scampi",
    verbose=False,
    scampi_delta=0.1
)



dists, candidates, elbow_points = scampi_last_two.fit_k_elbow(
    k_max=k_max,
    motif_length=motif_length,
    top_N=top_N,
    scampi_top_n_strategy="mask",
    plot_elbows=False,
    plot_motifs_as_grid=False,
)


In [ ]:
plot_topn(
    scampi_last_two,
    changepoints=changepoints[-3:]-changepoints[-3],
    annotations=annotations[-2:],
    fname=f"human-activity-w{scampi_last_two.motif_length}-last-two.pdf"
)

In [ ]:
def quantitative_evaluation(
    topn: SCAMPI,
    changepoints: np.array,
    annotations
):
    w = topn.motif_length
    section_motiflets = dict()

    for rank, elbows in enumerate(topn.elbow_points):
      if len(elbows) == 0:
          continue

      k = elbows[0]  # matches old motif_sets[elbow_points[-1]][0][0]
      hsm = np.asarray(topn.motiflets[k][rank]).squeeze()
      if hsm.size == 0:
          continue

      mlabels = dict()
      for t in hsm:
          t = int(t)
          segment = np.searchsorted(changepoints, t) - 1
          l = annotations[segment]
          mlabels[l] = mlabels.get(l, 0) + 1

      best_count, best_label = max((v, k) for k, v in mlabels.items())

      section = section_motiflets.get(best_label, [])
      section.extend(hsm)
      section_motiflets[best_label] = section

    results = []
    for section, motiflets in section_motiflets.items():
      total = 0
      covered = 0
      flags = np.zeros_like(topn.series)

      for t in motiflets:
          t = int(t)
          flags[t:t + w] = 1

      for l, (b, e) in zip(annotations, zip(changepoints[:-1], changepoints[1:])):
          if l == section:
              covered += flags[b:e].sum()
              total += e - b

      results.append(dict(section=section, total=total, covered=covered))

    results = pd.DataFrame(results)
    results["fraction"] = results["covered"] / results["total"]
    return results

In [ ]:
quantitative_evaluation(scampi_topn, changepoints, annotations)

In [ ]:
quantitative_evaluation(scampi_long_topn, changepoints, annotations)

In [ ]:
quantitative_evaluation(scampi_last_two, changepoints, annotations)

In [ ]:
find_dominant_window_sizes(series)

In [ ]:
find_dominant_window_sizes2(series[changepoints[-3]:])

In [ ]:
motif_length = 4 * find_dominant_window_sizes2(series)

long_topn_alt = SCAMPI(
    "Human Activity",
    series,
    n_jobs=-1,
    backend="scampi",
    verbose=False,
    scampi_delta=0.1,
)

dists, candidates, elbow_points = long_topn_alt.fit_k_elbow(
    k_max=100,
    motif_length=motif_length,
    top_N=12,
    scampi_top_n_strategy="mask",
    plot_elbows=False,
    plot_motifs_as_grid=False,
)


In [ ]:
plot_topn(
    long_topn_alt,
    changepoints,
    annotations,
    fname=f"human-activity-w{long_topn_alt.motif_length}.pdf"
)

In [ ]:
last_two_alt_topn = SCAMPI(
    "Human Activity",
    last_two,
    n_jobs=-1,
    backend="scampi",
    verbose=False,
    scampi_delta=0.1,
)

dists, candidates, elbow_points = last_two_alt_topn.fit_k_elbow(
    k_max=100,
    motif_length=3 * find_dominant_window_sizes2(last_two),
    top_N=3,
    scampi_top_n_strategy="mask",
    plot_elbows=False,
    plot_motifs_as_grid=False,
)



In [ ]:
plot_topn(last_two_alt_topn, changepoints=changepoints[-3:]-changepoints[-3], annotations=annotations[-2:], fname=f"human-activity-w{last_two_alt_topn.window_size}-last-two.pdf")

In [ ]:
5*find_dominant_window_sizes2(series)

In [ ]:
5*find_dominant_window_sizes2(series[changepoints[-3]:])